<a href="https://colab.research.google.com/github/aashishneupanecs24-creator/BIOS-LAB-1BF24CS004/blob/main/Parallel_Cellulart_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
from concurrent.futures import ThreadPoolExecutor

# ---------------------------------------------------------
# 1. Problem Definition: Task Dependency Graph (DAG)
# ---------------------------------------------------------
# 8 tasks with execution times (in ms)
TASK_TIMES = {
    0: 10,
    1: 15,
    2: 20,
    3: 12,
    4: 18,
    5: 14,
    6: 16,
    7: 10,
}

# Directed edges: (parent_task, child_task, communication_cost)
DEPENDENCIES = [
    (0, 1, 5),
    (0, 2, 6),
    (0, 3, 4),
    (1, 4, 7),
    (1, 5, 3),
    (2, 5, 5),
    (3, 6, 6),
    (4, 7, 4),
    (5, 7, 5),
    (6, 7, 4),
]

NUM_TASKS = len(TASK_TIMES)
NUM_PROCESSORS = 3  # Target parallel processors: P0, P1, P2


# Build topological order and cellular neighborhoods (parents + children)
def build_graph_info():
    parents = {t: [] for t in range(NUM_TASKS)}
    children = {t: [] for t in range(NUM_TASKS)}
    for u, v, cost in DEPENDENCIES:
        parents[v].append((u, cost))
        children[u].append((v, cost))
    return parents, children


PARENTS, CHILDREN = build_graph_info()


def calculate_makespan(allocation):
    """
    Computes total execution time (makespan) for a given cell state array,
    accounting for processor availability and inter-processor communication delays.
    """
    proc_free_time = [0] * NUM_PROCESSORS
    task_finish_time = {}

    # Tasks 0..7 are already topologically ordered
    for task in range(NUM_TASKS):
        assigned_proc = allocation[task]

        # Wait for all parent tasks (plus communication cost if on a different processor)
        data_ready_time = 0
        for parent, comm_cost in PARENTS[task]:
            delay = 0 if allocation[parent] == assigned_proc else comm_cost
            data_ready_time = max(
                data_ready_time, task_finish_time[parent] + delay
            )

        start_time = max(proc_free_time[assigned_proc], data_ready_time)
        finish_time = start_time + TASK_TIMES[task]

        proc_free_time[assigned_proc] = finish_time
        task_finish_time[task] = finish_time

    return max(task_finish_time.values())


# ---------------------------------------------------------
# 2. Parallel Cellular Algorithm (Local Transition Rule)
# ---------------------------------------------------------
def evaluate_cell_transition(args):
    """
    Local transition rule executed independently by each cell (task).
    Inspects neighbor states and picks the processor that minimizes local makespan.
    """
    cell_idx, current_states, mutation_prob = args
    best_proc = current_states[cell_idx]
    best_cost = calculate_makespan(current_states)

    # Explore candidate states based on local neighborhood
    for candidate_proc in range(NUM_PROCESSORS):
        trial_states = list(current_states)
        trial_states[cell_idx] = candidate_proc
        cost = calculate_makespan(trial_states)

        if cost < best_cost:
            best_cost = cost
            best_proc = candidate_proc

    # Small stochastic perturbation to escape local optima
    if random.random() < mutation_prob:
        best_proc = random.randint(0, NUM_PROCESSORS - 1)

    return best_proc


def parallel_cellular_scheduler(generations=50, mutation_prob=0.1):
    # Initialize random processor assignment for each cell
    states = [random.randint(0, NUM_PROCESSORS - 1) for _ in range(NUM_TASKS)]
    best_allocation = list(states)
    best_makespan = calculate_makespan(states)

    with ThreadPoolExecutor(max_workers=NUM_TASKS) as executor:
        for gen in range(generations):
            # Decay exploration rate over generations
            current_mut = mutation_prob * (1.0 - gen / generations)

            # Synchronous parallel update of all cells
            tasks = [(i, states, current_mut) for i in range(NUM_TASKS)]
            next_states = list(executor.map(evaluate_cell_transition, tasks))

            # Accept global improvement or update state
            new_makespan = calculate_makespan(next_states)
            if new_makespan <= best_makespan:
                best_makespan = new_makespan
                best_allocation = list(next_states)

            states = next_states

    return best_allocation, best_makespan


# ---------------------------------------------------------
# 3. Run Optimization
# ---------------------------------------------------------
if __name__ == "__main__":
    random.seed(42)
    sequential_time = sum(TASK_TIMES.values())
    best_schedule, min_makespan = parallel_cellular_scheduler(generations=60)

    print(f"Sequential Execution Time (1 CPU): {sequential_time} ms")
    print(f"Optimal Makespan ({NUM_PROCESSORS} CPUs):        {min_makespan} ms")
    print(
        f"Speedup Achieved:                  {sequential_time / min_makespan:.2f}x\n"
    )
    print("Task -> Assigned Processor:")
    for task_id, proc_id in enumerate(best_schedule):
        print(
            f"  Task {task_id} (duration {TASK_TIMES[task_id]} ms) -> Processor P{proc_id}"
        )

Sequential Execution Time (1 CPU): 115 ms
Optimal Makespan (3 CPUs):        65 ms
Speedup Achieved:                  1.77x

Task -> Assigned Processor:
  Task 0 (duration 10 ms) -> Processor P1
  Task 1 (duration 15 ms) -> Processor P0
  Task 2 (duration 20 ms) -> Processor P1
  Task 3 (duration 12 ms) -> Processor P2
  Task 4 (duration 18 ms) -> Processor P2
  Task 5 (duration 14 ms) -> Processor P0
  Task 6 (duration 16 ms) -> Processor P1
  Task 7 (duration 10 ms) -> Processor P2
